# Multiplication des pdf d'émission avec celle de la marée

In [ ]:
!pip install xdggs -U

In [1]:
import fsspec
import s3fs
import numpy as np
import xdggs
import xarray as xr
import pandas as pd

In [2]:
states = xr.open_dataset("/home/jovyan/pangeo-fish/notebooks/nouveau_tag/A15789/states_sigmas_30_100.zarr")

In [3]:
states.states.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(alpha=0.8)

MapWithSliders(children=(Map(custom_attribution='', layers=(SolidPolygonLayer(filled=True, get_fill_color=arro…

In [3]:
storage_options = {
    "anon": False,
    "profile": "gfts",
    "client_kwargs": {
        "endpoint_url": "https://s3.gra.perf.cloud.ovh.net",
        "region_name": "gra",
    },
}

fs = s3fs.S3FileSystem(
    anon=storage_options["anon"],
    profile=storage_options["profile"],
    client_kwargs=storage_options["client_kwargs"],
)

tag_name = "A15789"
scratch_root = "s3://gfts-ifremer/eel/run/chue/test"
target_root = f"{scratch_root}/{tag_name}"

In [4]:
fs.ls(target_root)

['gfts-ifremer/eel/run/chue/test/A15789/bathy_pdf.zarr',
 'gfts-ifremer/eel/run/chue/test/A15789/bathy_pdf_A15789.zarr',
 'gfts-ifremer/eel/run/chue/test/A15789/combined.zarr',
 'gfts-ifremer/eel/run/chue/test/A15789/diff-regridded.zarr',
 'gfts-ifremer/eel/run/chue/test/A15789/diff-regridded_bathy.zarr',
 'gfts-ifremer/eel/run/chue/test/A15789/diff.zarr',
 'gfts-ifremer/eel/run/chue/test/A15789/diff_bathy.zarr',
 'gfts-ifremer/eel/run/chue/test/A15789/emission.zarr',
 'gfts-ifremer/eel/run/chue/test/A15789/emission_w_bathy_pdf_A15789.zarr',
 'gfts-ifremer/eel/run/chue/test/A15789/emission_w_tide_A15789.zarr',
 'gfts-ifremer/eel/run/chue/test/A15789/parameters.json',
 'gfts-ifremer/eel/run/chue/test/A15789/states.zarr',
 'gfts-ifremer/eel/run/chue/test/A15789/tags.html']

**Chargement de emission DST**

In [5]:
emission = xr.open_dataset(f"{target_root}/emission_w_bathy_pdf_{tag_name}.zarr", engine="zarr")  # .compute()
emission.longitude.compute()
emission.latitude.compute()
emission

<xarray.Dataset> Size: 7GB
Dimensions:     (cells: 613119, time: 1441)
Coordinates:
    cell_ids    (cells) int64 5MB ...
    latitude    (cells) float64 5MB ...
    longitude   (cells) float64 5MB ...
    resolution  float64 8B ...
  * time        (time) datetime64[ns] 12kB 2019-12-17T17:00:00 ... 2020-02-15...
Dimensions without coordinates: cells
Data variables:
    final       (cells) float64 5MB ...
    initial     (cells) float64 5MB ...
    mask        (cells) float64 5MB ...
    pdf         (time, cells) float64 7GB ...
Attributes: (12/20)
    acoustic_tag_id:           NA
    comment:                   pangeo-fish == 2025.3.4.dev21+g331d0996b.d2025...
    common_name:               anguille
    differences_std:           0.75
    grid_type:                 healpix
    history:                   {'Conventions': 'CF-1.7', 'contact': 'serviced...
    ...                        ...
    relative_depth_threshold:  0
    rot_lat:                   0
    rot_lon:                   0
    scientific_name:           Unknow
    tag_name:                  A15789
    tagging_campaign:          JP

In [9]:
emission_tide.pdf.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(alpha=0.8)

MapWithSliders(children=(Map(custom_attribution='', layers=(SolidPolygonLayer(filled=True, get_fill_color=arro…

**Chargement de emission tide**
On adapte l'échelle temporelle et on récupère la variable "tide found" avant le merge pour éviter de la multiplier lors du combine. 

In [8]:
emission_tide = xr.open_dataset("/home/jovyan/pangeo-fish-old/notebooks/Tide/A15789_tide_pdf_healpix.zarr", engine="zarr")
emission_tide["pdf_tide"] = emission_tide["pdf"]

origin = pd.to_datetime(emission["time"].isel(time=0).values)
delta = pd.to_timedelta(emission_tide.time.values, unit="h")
emission_tide = emission_tide.assign_coords(time=origin + delta)

tide_found = emission_tide["tide_found"]
emission_tide = emission_tide.drop_vars("tide_found")
emission_tide

<xarray.Dataset> Size: 14GB
Dimensions:     (cells: 611815, time: 1441)
Coordinates:
    cell_ids    (cells) int64 5MB ...
    latitude    (cells) float64 5MB ...
    longitude   (cells) float64 5MB ...
    resolution  float64 8B ...
  * time        (time) datetime64[ns] 12kB 2019-12-17T17:00:00 ... 2020-02-15...
Dimensions without coordinates: cells
Data variables:
    ocean_mask  (cells) float64 5MB ...
    pdf         (time, cells) float64 7GB ...
    pdf_tide    (time, cells) float64 7GB ...
Attributes:
    comment:       pangeo-fish == 2025.3.4.dev11+g3e5801148.d20251201, healpi...
    grid_type:     healpix
    lat:           0
    level:         12
    lon:           0
    min_vertices:  1
    nside:         4096
    rot_lat:       0
    rot_lon:       0

## On sélectionne les cellules communes 
Les deux pdf n'ont pas les mêmes bbox de base, il faut donc sélectionner la bbox commune. 

In [8]:
common_cells = np.intersect1d(emission["cell_ids"], emission_tide["cell_ids"])
common_cells.shape

(611787,)

On crée les deux sous datasets que l'on va merge pour éviter les effets de bords. 

*Il y a peut être un moyen d'efectuer les changement directement sur les datasets originels pour éviter de prendre trop de mémoire.*

In [9]:
emission_small = emission.set_index(cells="cell_ids").sel(cells=common_cells)
emission_tide_small = emission_tide.set_index(cells="cell_ids").sel(cells=common_cells)
emission_small["dst_pdf"] = emission_small["pdf"]
emission_small

<xarray.Dataset> Size: 14GB
Dimensions:     (cells: 611787, time: 1441)
Coordinates:
    latitude    (cells) float64 5MB ...
    longitude   (cells) float64 5MB ...
    resolution  float64 8B ...
  * time        (time) datetime64[ns] 12kB 2019-12-17T17:00:00 ... 2020-02-15...
  * cells       (cells) int64 5MB 11237215 11237231 ... 58565156 58565160
Data variables:
    final       (cells) float64 5MB ...
    initial     (cells) float64 5MB ...
    mask        (cells) float64 5MB ...
    pdf         (time, cells) float64 7GB ...
    dst_pdf     (time, cells) float64 7GB ...
Attributes: (12/20)
    acoustic_tag_id:           NA
    comment:                   pangeo-fish == 2025.3.4.dev21+g331d0996b.d2025...
    common_name:               anguille
    differences_std:           0.75
    grid_type:                 healpix
    history:                   {'Conventions': 'CF-1.7', 'contact': 'serviced...
    ...                        ...
    relative_depth_threshold:  0
    rot_lat:                   0
    rot_lon:                   0
    scientific_name:           Unknow
    tag_name:                  A15789
    tagging_campaign:          JP

## On merge les deux datasets puis on combine les deux pdf 

On ajoute aussi la variable tide_found pour le multi-sigmas

Attention lorsqu'on utilise normalize pdf pour faire le combine, on multiplie toute les variables entre elles dont **tide_found**, on obtient donc une carte de 0 qui est ensuite transformée en carte de nan dans normalize_pdf.

**Bien rajouté tide_found après le normalize/combine**

In [10]:
from pangeo_fish.helpers import normalize_pdf

emission_with_tide = emission_tide_small.drop_vars("pdf_tide").merge(
    emission_small.drop_vars("pdf")
)
default_chunk_dims = {"time": 24}

/srv/conda/envs/notebook/lib/python3.12/site-packages/movingpandas/__init__.py:41: UserWarning: Missing optional dependencies. To use the trajectory smoother classes please install Stone Soup (see https://stonesoup.readthedocs.io/en/latest/#installation).
  warnings.warn(e.msg, UserWarning)


In [11]:
combined_diff_bathy = normalize_pdf(
    ds=emission_with_tide,
    chunks=default_chunk_dims,
    dims=["cells"],
)[0]
combined_diff_bathy
combined_diff_bathy = combined_diff_bathy.assign_coords(
    cell_ids=("cells", combined_diff_bathy["cells"].values)
)  # use cell_ids instead of cells
combined_diff_bathy["tide_found"] = tide_found  # add of tide_found variable
combined_diff_bathy

# emission_with_tide = emission_tide_small.merge(emission_small.drop_vars('pdf'))
# emission_with_tide = emission_with_tide.assign_coords(cell_ids=("cells", emission_with_tide["cells"].values))

<xarray.Dataset> Size: 7GB
Dimensions:     (cells: 611787, time: 1441)
Coordinates:
    latitude    (cells) float64 5MB dask.array<chunksize=(611787,), meta=np.ndarray>
    longitude   (cells) float64 5MB dask.array<chunksize=(611787,), meta=np.ndarray>
    resolution  float64 8B 0.0002498
  * cells       (cells) int64 5MB 11237215 11237231 ... 58565156 58565160
  * time        (time) datetime64[ns] 12kB 2019-12-17T17:00:00 ... 2020-02-15...
    cell_ids    (cells) int64 5MB 11237215 11237231 ... 58565156 58565160
Data variables:
    initial     (cells) float64 5MB dask.array<chunksize=(611787,), meta=np.ndarray>
    final       (cells) float64 5MB dask.array<chunksize=(611787,), meta=np.ndarray>
    mask        (cells) float64 5MB dask.array<chunksize=(611787,), meta=np.ndarray>
    pdf         (cells, time) float64 7GB dask.array<chunksize=(611787, 24), meta=np.ndarray>
    tide_found  (time) int64 12kB ...
Attributes:
    comment:       pangeo-fish == 2025.3.4.dev21+g331d0996b.d20251208, healpi...
    grid_type:     healpix
    lat:           0
    level:         12
    lon:           0
    min_vertices:  1
    nside:         4096
    rot_lat:       0
    rot_lon:       0

**Plot of the combine pdf**

Multiple visualisations 

In [ ]:
emission_with_tide.pdf.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(alpha=0.8) | emission_with_tide.pdf_tide.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(
    alpha=0.8
) | emission_with_tide.pdf_final.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(
    alpha=0.8
)

### Saving of the combined dataset

In [13]:
storage_options = {
    "anon": False,
    "profile": "gfts",
    "client_kwargs": {
        "endpoint_url": "https://s3.gra.perf.cloud.ovh.net",
        "region_name": "gra",
    },
}

combined_diff_bathy.compute().to_zarr(
    f"{target_root}/emission_w_tide_{tag_name}.zarr",
    compute=True,
    mode="w",
    consolidated=True,
    zarr_version=2,
    storage_options=storage_options,
)

/tmp/ipykernel_1260/3470634876.py:10: FutureWarning: zarr_version is deprecated, use zarr_format
  combined_diff_bathy.compute().to_zarr(


In [7]:

combined_diff_bathy = xr.open_dataset(
    f"{target_root}/emission_w_tide_{tag_name}.zarr", engine="zarr"
)
combined_diff_bathy

<xarray.Dataset> Size: 7GB
Dimensions:     (cells: 611787, time: 1441)
Coordinates:
    cell_ids    (cells) int64 5MB ...
  * cells       (cells) int64 5MB 11237215 11237231 ... 58565156 58565160
    latitude    (cells) float64 5MB ...
    longitude   (cells) float64 5MB ...
    resolution  float64 8B ...
  * time        (time) datetime64[ns] 12kB 2019-12-17T17:00:00 ... 2020-02-15...
Data variables:
    final       (cells) float64 5MB ...
    initial     (cells) float64 5MB ...
    mask        (cells) float64 5MB ...
    pdf         (cells, time) float64 7GB ...
    tide_found  (time) int64 12kB ...
Attributes:
    comment:       pangeo-fish == 2025.3.4.dev11+g3e5801148.d20251203, healpi...
    grid_type:     healpix
    lat:           0
    level:         12
    lon:           0
    min_vertices:  1
    nside:         4096
    rot_lat:       0
    rot_lon:       0

In [12]:
combined_diff_bathy.pdf.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(alpha=0.8)

MapWithSliders(children=(Map(custom_attribution='', layers=(SolidPolygonLayer(filled=True, get_fill_color=arro…

# Changement du sigma 

In [ ]:
def sigma_to_speed_kmh(
    sigma, earth_radius_km=6371, truncate=4, adjustment_factor=5, timedelta_h=1
):
    if sigma in (None, "None", np.nan):
        return np.nan
    try:
        sigma = float(sigma)
    except (ValueError, TypeError):
        return np.nan
    return sigma * (earth_radius_km / (timedelta_h))


def speed_kmh_to_sigma(speed, earth_radius_km=6371, timedelta_h=1):
    if speed in (None, "None", np.nan):
        return np.nan
    try:
        speed = float(speed)
    except (ValueError, TypeError):
        return np.nan
    return speed / (earth_radius_km / (timedelta_h))

In [ ]:
sigma_slow = speed_kmh_to_sigma(30 / 24)
sigma_fast = speed_kmh_to_sigma(100 / 24)
print("sigma_slow", sigma_slow)
print("sigma_fast", sigma_fast)

# Test bathy_pdf

In [ ]:
emission_bathy = xr.open_dataset(
    f"{target_root}/emission_w_bathy_pdf_A15789.zarr", engine="zarr"
)
emission_bathy

In [ ]:
emission_bathy.pdf.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(alpha=0.8)